In [11]:
# 导入库
# Cell 1: 环境设置与依赖导入
import argparse
import os
import numpy as np
import math
import random
import datetime
import time
import h5py  # 用于读取.h5文件
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torch.optim as optim
from einops import rearrange, reduce, repeat
from einops.layers.torch import Rearrange, Reduce
import matplotlib.pyplot as plt

# 设置随机种子和GPU
seed = 2021
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

gpus = [0]  # 根据您的GPU情况修改
os.environ['CUDA_VISIBLE_DEVICES'] = ','.join(map(str, gpus))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [12]:
# Cell 2: 模型定义 (与您提供的代码基本一致，但做了适配性调整)
class PatchEmbedding(nn.Module):
    def __init__(self, emb_size=40):
        super().__init__()
        # 移除了未使用的eegnet
        self.shallownet = nn.Sequential(
            nn.Conv2d(1, 40, (1, 25), (1, 1)),
            nn.Conv2d(40, 40, (62, 1), (1, 1)),
            nn.BatchNorm2d(40),
            nn.ELU(),
            nn.AvgPool2d((1, 75), (1, 15)),
            nn.Dropout(0.5),
        )
        self.projection = nn.Sequential(
            nn.Conv2d(40, emb_size, (1, 1), stride=(1, 1)),
            Rearrange('b e (h) (w) -> b (h w) e'),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.shallownet(x)
        x = self.projection(x)
        return x

class MultiHeadAttention(nn.Module):
    def __init__(self, emb_size, num_heads, dropout):
        super().__init__()
        self.emb_size = emb_size
        self.num_heads = num_heads
        self.keys = nn.Linear(emb_size, emb_size)
        self.queries = nn.Linear(emb_size, emb_size)
        self.values = nn.Linear(emb_size, emb_size)
        self.att_drop = nn.Dropout(dropout)
        self.projection = nn.Linear(emb_size, emb_size)

    def forward(self, x: torch.Tensor, mask: torch.Tensor = None) -> torch.Tensor:
        queries = rearrange(self.queries(x), "b n (h d) -> b h n d", h=self.num_heads)
        keys = rearrange(self.keys(x), "b n (h d) -> b h n d", h=self.num_heads)
        values = rearrange(self.values(x), "b n (h d) -> b h n d", h=self.num_heads)
        energy = torch.einsum('bhqd, bhkd -> bhqk', queries, keys)
        if mask is not None:
            fill_value = torch.finfo(torch.float32).min
            energy = energy.masked_fill(~mask, fill_value)
        scaling = self.emb_size ** (1 / 2)
        att = F.softmax(energy / scaling, dim=-1)
        att = self.att_drop(att)
        out = torch.einsum('bhal, bhlv -> bhav ', att, values)
        out = rearrange(out, "b h n d -> b n (h d)")
        out = self.projection(out)
        return out

class ResidualAdd(nn.Module):
    def __init__(self, fn):
        super().__init__()
        self.fn = fn
    def forward(self, x, **kwargs):
        res = x
        x = self.fn(x, **kwargs)
        x += res
        return x

class FeedForwardBlock(nn.Sequential):
    def __init__(self, emb_size, expansion, drop_p):
        super().__init__(
            nn.Linear(emb_size, expansion * emb_size),
            nn.GELU(),
            nn.Dropout(drop_p),
            nn.Linear(expansion * emb_size, emb_size),
        )

class TransformerEncoderBlock(nn.Sequential):
    def __init__(self, emb_size, num_heads=5, drop_p=0.5, forward_expansion=4, forward_drop_p=0.5):
        super().__init__(
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                MultiHeadAttention(emb_size, num_heads, drop_p),
                nn.Dropout(drop_p)
            )),
            ResidualAdd(nn.Sequential(
                nn.LayerNorm(emb_size),
                FeedForwardBlock(emb_size, expansion=forward_expansion, drop_p=forward_drop_p),
                nn.Dropout(drop_p)
            ))
        )

class TransformerEncoder(nn.Sequential):
    def __init__(self, depth, emb_size):
        super().__init__(*[TransformerEncoderBlock(emb_size) for _ in range(depth)])

class ClassificationHead(nn.Sequential):
    def __init__(self, emb_size, n_classes):
        super().__init__()
        # 简化的分类头，使用您原始代码中有效的fc部分
        self.fc = nn.Sequential(
            nn.Linear(190 * emb_size, 32),  # 190 * 40=7600 -> 32
            nn.ELU(),
            nn.Dropout(0.3),
            nn.Linear(32, n_classes)
        )

    def forward(self, x):
        # x shape: [batch, num_patches, emb_size]
        x = x.contiguous().view(x.size(0), -1)  # Flatten: [batch, 190 * 40]
        out = self.fc(x)
        return x, out  # 返回特征和分类结果

class ViT(nn.Sequential):
    def __init__(self, emb_size=40, depth=6, n_classes=3, **kwargs):  # n_classes 改为 3
        super().__init__(
            PatchEmbedding(emb_size),
            TransformerEncoder(depth, emb_size),
            ClassificationHead(emb_size, n_classes)
        )

In [13]:
# Cell 3: 修改后的数据加载与预处理函数
import h5py
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader, TensorDataset

def load_h5_data(file_path, has_label=False):
    """
    加载.h5文件，自动检测数据结构。
    支持多种可能的键名。
    """
    with h5py.File(file_path, 'r') as f:
        # 尝试不同的数据键名
        data_keys = ['x', 'data', 'X', 'EEG', 'eeg', 'features']
        label_keys = ['y', 'label', 'Y', 'labels', 'target']
        
        # 查找数据
        data = None
        for key in data_keys:
            if key in f:
                data = f[key][:]
                print(f"  找到数据，键名: '{key}'，形状: {data.shape}")
                break
        
        if data is None:
            # 如果顶层没有找到，检查是否在子组中
            for group_name in f.keys():
                if isinstance(f[group_name], h5py.Group):
                    for key in data_keys:
                        if key in f[group_name]:
                            data = f[group_name][key][:]
                            print(f"  找到数据，位置: '{group_name}/{key}'，形状: {data.shape}")
                            break
                    if data is not None:
                        break
        
        if data is None:
            raise ValueError(f"在文件 {file_path} 中找不到数据。可用的键: {list(f.keys())}")
        
        if has_label:
            # 查找标签
            labels = None
            for key in label_keys:
                if key in f:
                    labels = f[key][:]
                    print(f"  找到标签，键名: '{key}'，形状: {labels.shape}")
                    break
            
            if labels is None:
                # 如果顶层没有找到，检查是否在子组中
                for group_name in f.keys():
                    if isinstance(f[group_name], h5py.Group):
                        for key in label_keys:
                            if key in f[group_name]:
                                labels = f[group_name][key][:]
                                print(f"  找到标签，位置: '{group_name}/{key}'，形状: {labels.shape}")
                                break
                        if labels is not None:
                            break
            
            if labels is None:
                raise ValueError(f"在文件 {file_path} 中找不到标签。可用的键: {list(f.keys())}")
            
            # 确保标签是1D数组
            if labels.ndim > 1:
                labels = labels.squeeze()
            
            # 标签转换 (如果原始标签是1,2,3，转换为0,1,2)
            unique_labels = np.unique(labels)
            print(f"  原始标签值: {unique_labels}")
            if set(unique_labels) == {1, 2, 3}:
                labels = labels - 1
                print("  已将标签从[1,2,3]转换为[0,1,2]")
            elif set(unique_labels) == {0, 1, 2}:
                print("  标签已经是[0,1,2]格式")
            else:
                print(f"  警告: 标签值域为 {unique_labels}，预期为 {{0,1,2}} 或 {{1,2,3}}")
            
            return data, labels
        else:
            return data

def preprocess_data(train_data, val_data=None, test_data=None):
    """
    数据预处理：标准化。
    使用训练集的均值和标准差来标准化所有数据。
    """
    # 计算训练集的均值和标准差
    # 假设数据形状: (样本数, 通道数, 时间点) 或 (样本数, 1, 通道数, 时间点)
    if train_data.ndim == 3:
        # (样本数, 通道数, 时间点)
        mean = np.mean(train_data, axis=(0, 2), keepdims=True)  # 按样本和时间点平均，保持通道维度
        std = np.std(train_data, axis=(0, 2), keepdims=True)
    elif train_data.ndim == 4:
        # (样本数, 1, 通道数, 时间点)
        mean = np.mean(train_data, axis=(0, 3), keepdims=True)  # 按样本和时间点平均
        std = np.std(train_data, axis=(0, 3), keepdims=True)
    else:
        raise ValueError(f"数据维度 {train_data.ndim} 不支持。期望 3 或 4 维。")
    
    print(f"  标准化 - 均值: {mean.shape}, 标准差: {std.shape}")
    
    train_data_norm = (train_data - mean) / (std + 1e-8)
    
    if val_data is not None:
        val_data_norm = (val_data - mean) / (std + 1e-8)
    else:
        val_data_norm = None
        
    if test_data is not None:
        test_data_norm = (test_data - mean) / (std + 1e-8)
    else:
        test_data_norm = None
        
    return train_data_norm, val_data_norm, test_data_norm, mean, std

def prepare_dataloader(data, labels=None, batch_size=64, shuffle=True):
    """
    创建PyTorch DataLoader。
    """
    # 确保数据是float32类型
    data_tensor = torch.from_numpy(data).float()
    
    # 调整数据形状以匹配模型输入: [batch, 1, channels, time_points]
    if data_tensor.ndim == 3:
        # 假设是 [样本数, 通道数, 时间点]
        data_tensor = data_tensor.unsqueeze(1)  # -> [样本数, 1, 通道数, 时间点]
    # 如果已经是4维 [样本数, 1, 通道数, 时间点]，则保持不变
    
    print(f"  数据形状调整: {data.shape} -> {data_tensor.shape}")
    
    if labels is not None:
        labels_tensor = torch.from_numpy(labels).long()
        dataset = TensorDataset(data_tensor, labels_tensor)
    else:
        dataset = TensorDataset(data_tensor)  # 只有数据，没有标签
        
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=shuffle)
    return dataloader

In [14]:
# Cell 4: 训练与验证函数
def train_one_epoch(model, train_loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        
        optimizer.zero_grad()
        _, output = model(data)  # 模型返回特征和输出
        loss = criterion(output, target)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * data.size(0)
        _, predicted = output.max(1)
        total += target.size(0)
        correct += predicted.eq(target).sum().item()
    
    avg_loss = total_loss / total
    accuracy = 100. * correct / total
    return avg_loss, accuracy

def validate(model, val_loader, criterion, device):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for data, target in val_loader:
            data, target = data.to(device), target.to(device)
            _, output = model(data)
            loss = criterion(output, target)
            
            total_loss += loss.item() * data.size(0)
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()
    
    avg_loss = total_loss / total
    accuracy = 100. * correct / total
    return avg_loss, accuracy

In [15]:
# Cell 5: 主程序 - 使用新的数据加载函数
def main():
    # --- 配置参数 ---
    train_file = "data/SEED/train.h5"  # 请修改为您的实际路径
    val_file = "data/SEED/val.h5"      # 请修改为您的实际路径
    test_file = "data/SEED/test_x_only.h5"    # 请修改为您的实际路径
    
    output_pred_file = "test_predictions.txt"  # 预测结果输出文件
    best_model_path = "best_model_seed.pth"    # 最佳模型保存路径
    
    batch_size = 64
    num_epochs = 100
    learning_rate = 0.0002
    patience = 10  # 早停耐心值
    
    # --- 1. 加载数据 ---
    print("=== 加载数据 ===")
    train_data, train_labels = load_h5_data(train_file, has_label=True)
    val_data, val_labels = load_h5_data(val_file, has_label=True)
    test_data = load_h5_data(test_file, has_label=False)  # 无标签
    
    print(f"\n数据统计:")
    print(f"  训练集: {train_data.shape}, 标签: {train_labels.shape}")
    print(f"  验证集: {val_data.shape}, 标签: {val_labels.shape}")
    print(f"  测试集: {test_data.shape}")
    
    # 检查数据形状并调整
    # 您的原始模型输入期望: [batch, 1, 62, 时间点]
    # 如果数据是 [样本数, 62, 时间点]，则添加通道维度
    if train_data.ndim == 3:
        print("\n检测到3维数据，添加通道维度...")
        # 假设维度是 [样本数, 通道数, 时间点]
        # 我们需要 [样本数, 1, 通道数, 时间点]
        train_data = np.expand_dims(train_data, axis=1)
        val_data = np.expand_dims(val_data, axis=1)
        test_data = np.expand_dims(test_data, axis=1)
        print(f"  调整后训练集形状: {train_data.shape}")
    elif train_data.ndim == 4:
        print(f"\n检测到4维数据，形状: {train_data.shape}")
        # 检查是否已经是 [样本数, 1, 通道数, 时间点]
        if train_data.shape[1] != 1:
            print(f"  警告: 数据的第2维是 {train_data.shape[1]}，预期为 1")
    
    # --- 2. 数据预处理（标准化）---
    print("\n=== 数据预处理（标准化）===")
    train_data_norm, val_data_norm, test_data_norm, mean, std = preprocess_data(
        train_data, val_data, test_data
    )
    
    # --- 3. 创建数据加载器 ---
    print("\n=== 创建数据加载器 ===")
    train_loader = prepare_dataloader(train_data_norm, train_labels, batch_size=batch_size, shuffle=True)
    val_loader = prepare_dataloader(val_data_norm, val_labels, batch_size=batch_size, shuffle=False)
    
    # 测试集没有标签
    test_tensor = torch.from_numpy(test_data_norm).float()
    if test_tensor.ndim == 3:
        test_tensor = test_tensor.unsqueeze(1)
    test_dataset = TensorDataset(test_tensor)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    # --- 4. 初始化模型、损失函数、优化器 ---
    print("\n=== 初始化模型 ===")
    model = ViT(emb_size=40, depth=6, n_classes=3).to(device)
    
    if len(gpus) > 1 and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model, device_ids=list(range(len(gpus))))
    
    # 打印模型结构
    print(f"模型参数量: {sum(p.numel() for p in model.parameters()):,}")
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate, betas=(0.5, 0.999))
    
    # --- 5. 训练循环（带早停）---
    print(f"\n=== 开始训练，共 {num_epochs} 轮 ===")
    best_val_acc = 0.0
    patience_counter = 0
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []
    
    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        
        # 记录损失和准确率
        train_losses.append(train_loss)
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)
        
        print(f"Epoch {epoch+1:03d}/{num_epochs:03d} | "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
        
        # 保存最佳模型
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            patience_counter = 0
            torch.save(model.state_dict(), best_model_path)
            print(f"  ✓ 保存最佳模型，验证准确率: {val_acc:.2f}%")
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"\n早停于第 {epoch+1} 轮")
                break
    
    print(f"\n训练完成，最佳验证准确率: {best_val_acc:.2f}%")
    
    # --- 6. 在测试集上进行预测 ---
    print("\n=== 加载最佳模型进行测试集预测 ===")
    model.load_state_dict(torch.load(best_model_path))
    model.eval()
    
    all_predictions = []
    
    with torch.no_grad():
        for batch in test_loader:
            data = batch[0].to(device)  # batch[0] 因为只有数据，没有标签
            _, outputs = model(data)
            _, predicted = outputs.max(1)
            all_predictions.extend(predicted.cpu().numpy())
    
    # 确保预测值是0,1,2
    all_predictions = np.array(all_predictions, dtype=int)
    
    # --- 7. 保存预测结果 ---
    print(f"\n=== 保存预测结果到 {output_pred_file} ===")
    np.savetxt(output_pred_file, all_predictions, fmt='%d')
    
    # 打印预测分布
    unique, counts = np.unique(all_predictions, return_counts=True)
    print("预测类别分布:")
    for cls, count in zip(unique, counts):
        print(f"  类别 {cls}: {count} 个样本 ({count/len(all_predictions)*100:.1f}%)")
    
    return all_predictions, train_losses, val_losses, train_accs, val_accs

# 运行主程序
if __name__ == "__main__":
    start_time = time.time()
    predictions, train_losses, val_losses, train_accs, val_accs = main()
    end_time = time.time()
    print(f"\n总耗时: {end_time - start_time:.2f} 秒")

=== 加载数据 ===


  找到数据，键名: 'X'，形状: (900, 62, 400)
  找到标签，键名: 'y'，形状: (900,)
  原始标签值: [0 1 2]
  标签已经是[0,1,2]格式
  找到数据，键名: 'X'，形状: (450, 62, 400)
  找到标签，键名: 'y'，形状: (450,)
  原始标签值: [0 1 2]
  标签已经是[0,1,2]格式
  找到数据，键名: 'X'，形状: (450, 62, 400)

数据统计:
  训练集: (900, 62, 400), 标签: (900,)
  验证集: (450, 62, 400), 标签: (450,)
  测试集: (450, 62, 400)

检测到3维数据，添加通道维度...
  调整后训练集形状: (900, 1, 62, 400)

=== 数据预处理（标准化）===
  标准化 - 均值: (1, 1, 62, 1), 标准差: (1, 1, 62, 1)

=== 创建数据加载器 ===
  数据形状调整: (900, 1, 62, 400) -> torch.Size([900, 1, 62, 400])
  数据形状调整: (450, 1, 62, 400) -> torch.Size([450, 1, 62, 400])

=== 初始化模型 ===
模型参数量: 463,651

=== 开始训练，共 100 轮 ===


RuntimeError: mat1 and mat2 shapes cannot be multiplied (64x840 and 7600x32)

In [ ]:
# Cell 6: 可视化训练过程
import matplotlib.pyplot as plt

def plot_training_curves(train_losses, val_losses, train_accs, val_accs):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    
    # 损失曲线
    ax1.plot(train_losses, label='训练损失')
    ax1.plot(val_losses, label='验证损失')
    ax1.set_xlabel('轮次')
    ax1.set_ylabel('损失')
    ax1.set_title('训练和验证损失')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 准确率曲线
    ax2.plot(train_accs, label='训练准确率')
    ax2.plot(val_accs, label='验证准确率')
    ax2.set_xlabel('轮次')
    ax2.set_ylabel('准确率 (%)')
    ax2.set_title('训练和验证准确率')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# 绘制训练曲线
plot_training_curves(train_losses, val_losses, train_accs, val_accs)

In [ ]:
# Cell 7: 检查预测结果
def check_predictions(file_path="test_predictions.txt"):
    if os.path.exists(file_path):
        preds = np.loadtxt(file_path, dtype=int)
        print(f"从 '{file_path}' 加载了 {len(preds)} 个预测")
        print(f"前20个预测: {preds[:20]}")
        
        unique, counts = np.unique(preds, return_counts=True)
        print("\n类别分布:")
        for cls, count in zip(unique, counts):
            print(f"  类别 {cls}: {count} 个样本 ({count/len(preds)*100:.1f}%)")
        
        # 检查是否只有0,1,2三个类别
        if set(unique) <= {0, 1, 2}:
            print("✓ 预测结果只包含0,1,2三个类别")
        else:
            print(f"⚠ 警告: 预测结果包含非预期类别: {unique}")
    else:
        print(f"文件 '{file_path}' 不存在")

check_predictions()